# UC04 — báo cáo và phát lại ca thật

Notebook chỉ đọc predictions/checkpoint đã hoàn tất, không huấn luyện lại hoặc đổi ngưỡng.

In [ ]:
from pathlib import Path
import sys
import os

# Sửa SOURCE_ROOT nếu dùng source được gắn qua Kaggle Input.
SOURCE_ROOT = Path.cwd()
if not (SOURCE_ROOT / "src").is_dir() and (SOURCE_ROOT.parent / "src").is_dir():
    SOURCE_ROOT = SOURCE_ROOT.parent
# SOURCE_ROOT = Path("/kaggle/input/safeanes-source")
assert (SOURCE_ROOT / "src" / "safeanes").is_dir(), "Đặt SOURCE_ROOT tới repository"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else SOURCE_ROOT
sys.path.insert(0, str(SOURCE_ROOT / "src"))
# Không đưa .local_deps Windows sang Kaggle.
if os.name == "nt" and (SOURCE_ROOT / ".local_deps").is_dir():
    sys.path.insert(0, str(SOURCE_ROOT / ".local_deps"))
print("Source:", SOURCE_ROOT, "Output:", WORK_ROOT)


In [ ]:
from safeanes.reporting import build_report
DATASET = WORK_ROOT / "data/pilot_v1"
RUN = WORK_ROOT / "artifacts/tcn_v1"
OUT = WORK_ROOT / "reports/tcn_v1"
print(build_report(DATASET, RUN, OUT))


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))
for figure in sorted(OUT.glob("*.png")):
    display(Image(filename=str(figure)))


In [ ]:
import json
diagnostics = json.loads((OUT / "diagnostics.json").read_text(encoding="utf-8"))
for model, result in diagnostics.items():
    print(model, "calibration slope:", result["calibration"]["slope"],
          "intercept:", result["calibration"]["intercept"])
    print("Quality gates:", result["targets"])


Xem mẫu số biến cố, CI, abstention và alarm censored. Hình MAP dùng giá trị tại lưới quyết định 30 giây; audit onset chính xác cần trở lại raw 1 giây. Diagnostic slope/intercept trên pilot_test không được áp lại để cải thiện score.